## I. descargas en en la maquina
- Transformadores, huggin_face
- Librerias para la vectorización

In [1]:
!pip install -U transformers accelerate bitsandbytes huggingface_hub

In [11]:
!pip install chromadb sentence-transformers pymupdf

## II. importacion de librerias

In [13]:
import torch, json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata
from sentence_transformers import SentenceTransformer
import chromadb, fitz, os

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

## III. f(x) auxiliares

In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [4]:
model_id = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [7]:
bateria_preguntas = [
    "Explicame en tres parrafo la normativa de agricultura de la rioja",
    "Que plagas afectan directamente al viñedo",
    "cuales son las mejores bodegas de la rioja",
    "que vinos son mas vendidos en el exterior"
]

In [ ]:
embedding_model = SentenceTransformer("BAAI/bge-m3")
cliente_db = chromadb.PersistentClient(path="./vectordb_rioja")
coleccion = cliente_db.get_or_create_collection("normativa_agricultura")

## IV. codigo principal

In [8]:
for i, pregunta in enumerate(bateria_preguntas, 1):
    print(f"--- Prueba {i} ---")
    print(f"Usuario: {pregunta}")

    messages = [{"role": "user", "content": pregunta}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.3, # Temperatura baja/media para respuestas coherentes
        do_sample=True
    )

    input_length = inputs["input_ids"].shape[-1]
    respuesta = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    print(f"Gemma 4B: {respuesta}\n")

--- Prueba 1 ---
Usuario: Explicame en tres parrafo la normativa de agricultura de la rioja
Gemma 4B: La agricultura en la Rioja, reconocida mundialmente por la calidad de sus vinos, está fuertemente regulada por una normativa específica que busca proteger la identidad y reputación de la Denominación de Origen Rioja (DOR). Esta normativa se basa principalmente en la Ley de Viñedos de España y en el Reglamento de la Denominación de Origen Rioja, y se centra en asegurar la calidad y autenticidad de los productos elaborados en la región.  La principal preocupación es la protección de la "Rioja auténtica", que se define como la producción de vino que se realiza en la comarca de Rioja, utilizando uvas procedentes de viñedos de la misma zona, y siguiendo las técnicas tradicionales y las normas establecidas.

La normativa establece requisitos estrictos en diversas áreas, desde la elaboración del vino hasta su comercialización.  En cuanto a la elaboración, se exige el uso de uvas de la varieda

##  V. vectorización de pdfs

*   Cargamos pdf_limpio y vectorizado
*   Elemento de lista



In [ ]:
with open("/content/boletin_5_2025_chunks.json", "r") as f:
    datos = json.load(f)

embedding_model = SentenceTransformer("BAAI/bge-m3")
coleccion = chromadb.PersistentClient("./vectordb_rioja").get_or_create_collection("normativa")

textos = [c["texto"] for c in datos["chunks"]]
vectores = embedding_model.encode(textos, show_progress_bar=True).tolist()

coleccion.add(
    documents=textos,
    embeddings=vectores,
    metadatas=[{"fuente": c["fuente"], "pagina": c["pagina"]} for c in datos["chunks"]],
    ids=[f"{datos['archivo_origen']}_chunk_{i}" for i in range(len(textos))]
)